# ⛈️ Thunderstorm Forecasting System
## End-to-End ML Pipeline with MLflow (DagsHub) + Streamlit

**Project Structure:**
```
thunderstorm-forecasting/
├── data/
│   ├── raw/          ← indices data.csv, surface_data.csv
│   └── processed/    ← merged_df_all12k_combined.csv
├── models/           ← KNN_best_model.pkl, best_model_Random_Forest.pkl
├── streamlit_app/    ← ui.py  (served via ngrok)
├── api/              ← main.py (FastAPI)
├── experiments/      ← experiment.ipynb, mlruns/
└── outputs/          ← metrics, plots
```
**Stack:** Pandas · Scikit-learn · Imbalanced-learn (SMOTE) · MLflow · DagsHub · Streamlit · ngrok

## 📦 Step 0 — Install Dependencies

In [ ]:
# Install all required packages
!pip install -q mlflow dagshub scikit-learn imbalanced-learn \
              streamlit pyngrok pandas numpy matplotlib seaborn joblib fastapi uvicorn
print('✅ All packages installed')

## 🔑 Step 1 — Environment Variables from Colab Secrets
> Add your secrets via: **Colab left sidebar → Key icon → Add new secret**  
> Keys needed: `MONGO_DB_URL`, `MLFLOW_TRACKING_URI`, `MLFLOW_TRACKING_USERNAME`, `MLFLOW_TRACKING_PASSWORD`, `NGROK_AUTHTOKEN`

In [ ]:
import os
from google.colab import userdata

# ── MongoDB (optional — used for raw data ingestion) ──────────────
os.environ['MONGO_DB_URL'] = userdata.get('MONGO_DB_URL')

# ── MLflow / DagsHub tracking ─────────────────────────────────────
USE_DAGSHUB = True  # Set False to log locally to /content/mlruns instead

if USE_DAGSHUB:
    os.environ['MLFLOW_TRACKING_URI']      = userdata.get('MLFLOW_TRACKING_URI')
    os.environ['MLFLOW_TRACKING_USERNAME'] = userdata.get('MLFLOW_TRACKING_USERNAME')
    os.environ['MLFLOW_TRACKING_PASSWORD'] = userdata.get('MLFLOW_TRACKING_PASSWORD')
else:
    # Local MLflow: logs saved to /content/mlruns
    os.environ['MLFLOW_TRACKING_URI'] = f"file://{os.getcwd()}/mlruns"

print('✅ Env vars set')
print(f"   MLFLOW_TRACKING_URI = {os.environ['MLFLOW_TRACKING_URI']}")

## 📁 Step 2 — Project Directory Setup (Colab Local)

In [ ]:
import os

BASE_DIR = '/content/thunderstorm_forecasting'

dirs = [
    f'{BASE_DIR}/data/raw',
    f'{BASE_DIR}/data/processed',
    f'{BASE_DIR}/models',
    f'{BASE_DIR}/streamlit_app',
    f'{BASE_DIR}/api',
    f'{BASE_DIR}/outputs',
    f'{BASE_DIR}/experiments',
]

for d in dirs:
    os.makedirs(d, exist_ok=True)

print('✅ Project directories created:')
for d in dirs:
    print(f'   {d}')

## 📂 Step 3 — Upload Raw Data CSVs
Upload `indices data.csv` and `surface_data.csv` using the cell below, or mount Google Drive.

In [ ]:
from google.colab import files
import shutil

print('Upload the two raw CSV files:')
print('  1. indices data.csv')
print('  2. surface_data.csv')
uploaded = files.upload()

for fname in uploaded:
    dest = f"{BASE_DIR}/data/raw/{fname}"
    with open(dest, 'wb') as f:
        f.write(uploaded[fname])
    print(f'✅ Saved → {dest}')

### (Alternative) Mount Google Drive

In [ ]:
# Uncomment if your CSVs are on Drive
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy('/content/drive/MyDrive/YOUR_PATH/indices data.csv', f'{BASE_DIR}/data/raw/')
# shutil.copy('/content/drive/MyDrive/YOUR_PATH/surface_data.csv', f'{BASE_DIR}/data/raw/')
print('Skip this cell if you already uploaded files above.')

## 🔍 Step 4 — Exploratory Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Load raw data ─────────────────────────────────────────────────
df_indices = pd.read_csv(f'{BASE_DIR}/data/raw/indices data.csv')
df_surface = pd.read_csv(f'{BASE_DIR}/data/raw/surface_data.csv')

print('=== Atmospheric Indices Dataset ===')
print(f'Shape: {df_indices.shape}')
print(f'Columns: {df_indices.columns.tolist()}')
display(df_indices.head(3))

print('\n=== Surface Observations Dataset ===')
print(f'Shape: {df_surface.shape}')
print(f'Columns: {df_surface.columns.tolist()}')
display(df_surface.head(3))

print('\n=== Target (TH) Distribution in Surface Data ===')
print(df_surface['TH'].value_counts())

## 🔗 Step 5 — Data Merging & Preprocessing
The **indices data** (row index = Year, columns = Month/Day/Hour + indices) is merged with
**surface data** (INDEX=Year, YEAR=Month, MN=Day, TH=target) on a common Year-Month-Day key.

In [ ]:
# ── 5.1  Rename columns to consistent Year / Month / Day scheme ──
df_indices = df_indices.copy()
df_indices.index.name = 'Year'
df_indices = df_indices.reset_index()
df_indices = df_indices.rename(columns={'Year':'Year_','Year_':'Year',
                                         'index':'Year',
                                         'Month':'Day', 'Date':'Hour'})
# Safer rename using positional header knowledge
df_indices.columns = ['Year','Month','Day','Hour'] + list(df_indices.columns[4:])

df_surface = df_surface.rename(columns={'INDEX':'Year', 'YEAR':'Month', 'MN':'Day'})

print('Indices (renamed) head:')
display(df_indices[['Year','Month','Day','Hour']].head(3))
print('Surface (renamed) head:')
display(df_surface[['Year','Month','Day','TH']].head(3))

# ── 5.2  Aggregate indices to daily (mean of 00 & 12 UTC) ────────
idx_daily = (
    df_indices
    .groupby(['Year','Month','Day'], as_index=False)
    .mean(numeric_only=True)
)
print(f'\nIndices daily shape: {idx_daily.shape}')

# ── 5.3  Clean target column ──────────────────────────────────────
df_surface['TH'] = pd.to_numeric(df_surface['TH'], errors='coerce').fillna(0).astype(int)
surf_daily = df_surface[['Year','Month','Day','TH']].copy()

# ── 5.4  Merge ───────────────────────────────────────────────────
merged = idx_daily.merge(surf_daily, on=['Year','Month','Day'], how='inner')
print(f'Merged shape: {merged.shape}')
print(f'\nTarget distribution:\n{merged["TH"].value_counts()}')
display(merged.head(5))

In [ ]:
# Save processed (merged) dataset
processed_path = f'{BASE_DIR}/data/processed/merged_df_all12k_combined.csv'
merged.to_csv(processed_path, index=False)
print(f'✅ Processed data saved → {processed_path}')

## 📊 Step 6 — Visualisations

In [ ]:
feature_cols = ['SWEAT index','Showalter index','LIFTED index','K index',
                'Cross totals index','Vertical totals index','Totals totals index',
                'TLCL','PLCL','CINE','CAPE','PRECIPITABLE WATER']

# Drop rows with too many NaNs
df_clean = merged[feature_cols + ['TH']].dropna()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
key_features = ['K index','CAPE','SWEAT index','LIFTED index','Totals totals index','PRECIPITABLE WATER']
for ax, feat in zip(axes.flatten(), key_features):
    df_clean.boxplot(column=feat, by='TH', ax=ax)
    ax.set_title(feat)
    ax.set_xlabel('Thunderstorm (0=No, 1=Yes)')
plt.suptitle('Feature Distributions by Thunderstorm Occurrence')
plt.tight_layout()
plt.savefig(f'{BASE_DIR}/outputs/feature_boxplots.png', dpi=100)
plt.show()

# Correlation heatmap
fig, ax = plt.subplots(figsize=(12, 8))
corr = df_clean[feature_cols].corr()
sns.heatmap(corr, annot=True, fmt='.1f', cmap='coolwarm', ax=ax)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig(f'{BASE_DIR}/outputs/correlation_heatmap.png', dpi=100)
plt.show()

# Class imbalance pie
fig, ax = plt.subplots(figsize=(5, 5))
merged['TH'].value_counts().plot.pie(labels=['No TH','Thunderstorm'], autopct='%1.1f%%', ax=ax)
ax.set_title('Class Distribution (Before SMOTE)')
plt.tight_layout()
plt.savefig(f'{BASE_DIR}/outputs/class_distribution.png', dpi=100)
plt.show()

print('✅ Visualisations saved to outputs/')

## ⚙️ Step 7 — Feature Engineering & SMOTE

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

FEATURE_COLS = ['SWEAT index','Showalter index','LIFTED index','K index',
                'Cross totals index','Vertical totals index','Totals totals index',
                'TLCL','PLCL','CINE','CAPE','PRECIPITABLE WATER']
TARGET = 'TH'

# ── Clean ────────────────────────────────────────────────────────
df_model = merged[FEATURE_COLS + [TARGET]].dropna().copy()
X = df_model[FEATURE_COLS]
y = df_model[TARGET]
print(f'Dataset size before SMOTE: {X.shape}, TH distribution: {dict(y.value_counts())}')

# ── Train / Test split ───────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}')

# ── SMOTE on training set only ───────────────────────────────────
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f'After SMOTE — Train: {X_train_res.shape[0]}, TH dist: {dict(pd.Series(y_train_res).value_counts())}')

# ── Scale ────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled  = scaler.transform(X_test)

print('\n✅ Preprocessing complete')

## 🧪 Step 8 — Model Training + MLflow Experiment Tracking
Trains **Random Forest**, **KNN**, **Logistic Regression**, and **Decision Tree** — each logged
as a separate MLflow run. The best model is saved to `models/`.

In [ ]:
import mlflow
import mlflow.sklearn
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)

mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
mlflow.set_experiment('Thunderstorm_Forecasting')

MODELS = {
    'Random_Forest': RandomForestClassifier(
        n_estimators=200, max_depth=10, random_state=42, class_weight='balanced'
    ),
    'KNN': KNeighborsClassifier(
        n_neighbors=5, weights='distance'
    ),
    'Logistic_Regression': LogisticRegression(
        max_iter=500, class_weight='balanced', random_state=42
    ),
    'Decision_Tree': DecisionTreeClassifier(
        max_depth=8, random_state=42, class_weight='balanced'
    ),
}

results = {}

for model_name, model in MODELS.items():
    print(f'\n🔄 Training {model_name} ...')
    with mlflow.start_run(run_name=model_name):

        # ── Log hyperparameters ──────────────────────────────────
        mlflow.log_params(model.get_params())
        mlflow.log_param('smote', True)
        mlflow.log_param('scaler', 'StandardScaler')
        mlflow.log_param('features', FEATURE_COLS)
        mlflow.log_param('train_size', X_train_scaled.shape[0])
        mlflow.log_param('test_size',  X_test_scaled.shape[0])

        # ── Fit ──────────────────────────────────────────────────
        model.fit(X_train_scaled, y_train_res)
        y_pred = model.predict(X_test_scaled)

        # ── Metrics ──────────────────────────────────────────────
        acc  = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec  = recall_score(y_test, y_pred, zero_division=0)
        f1   = f1_score(y_test, y_pred, zero_division=0)
        try:
            auc = roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:,1])
        except Exception:
            auc = 0.0

        mlflow.log_metrics({'accuracy':acc,'precision':prec,'recall':rec,'f1':f1,'roc_auc':auc})

        # ── Confusion matrix plot ────────────────────────────────
        cm_path = f'{BASE_DIR}/outputs/cm_{model_name}.png'
        fig, ax = plt.subplots(figsize=(4,4))
        ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred)).plot(ax=ax)
        ax.set_title(model_name)
        plt.tight_layout()
        plt.savefig(cm_path)
        plt.close()
        mlflow.log_artifact(cm_path)

        # ── Log model ─────────────────────────────────────────────
        mlflow.sklearn.log_model(model, artifact_path='model')

        results[model_name] = {'acc':acc,'prec':prec,'rec':rec,'f1':f1,'auc':auc,'model':model}
        print(f'   Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  F1={f1:.4f}  AUC={auc:.4f}')

print('\n✅ All runs logged to MLflow')

## 🏆 Step 9 — Select & Save Best Model

In [ ]:
# ── Compare results ──────────────────────────────────────────────
results_df = pd.DataFrame({
    k: {m: v[m] for m in ['acc','prec','rec','f1','auc']}
    for k, v in results.items()
}).T
print('=== Model Comparison ===')
display(results_df.sort_values('f1', ascending=False))

# ── Save best by F1 score ────────────────────────────────────────
best_name = results_df['f1'].idxmax()
best_model = results[best_name]['model']
print(f'\n🏆 Best model: {best_name} (F1={results[best_name]["f1"]:.4f})')

# Save models
joblib.dump(best_model, f'{BASE_DIR}/models/best_model_{best_name}.pkl')
joblib.dump(scaler,     f'{BASE_DIR}/models/scaler.pkl')

# Also save KNN and RF explicitly (used in Streamlit app)
joblib.dump(results['Random_Forest']['model'], f'{BASE_DIR}/models/best_model_Random_Forest.pkl')
joblib.dump(results['KNN']['model'],           f'{BASE_DIR}/models/KNN_best_model.pkl')

print('✅ Models saved:')
for f in os.listdir(f'{BASE_DIR}/models'):
    print(f'   {BASE_DIR}/models/{f}')

## ✅ Step 10 — Prediction Sanity Check

In [ ]:
# Load and test
loaded_model  = joblib.load(f'{BASE_DIR}/models/best_model_Random_Forest.pkl')
loaded_scaler = joblib.load(f'{BASE_DIR}/models/scaler.pkl')

# Sample: known thunderstorm day (use actual +ve row from test set)
pos_indices = y_test[y_test == 1].index
if len(pos_indices) > 0:
    sample = X_test.loc[pos_indices[0]]
    print('Input features (thunderstorm day):')
    print(sample.to_dict())
    sample_scaled = loaded_scaler.transform([sample.values])
    pred = loaded_model.predict(sample_scaled)[0]
    prob = loaded_model.predict_proba(sample_scaled)[0]
    print(f'\nPrediction: {"⛈️ THUNDERSTORM" if pred == 1 else "☀️ NO THUNDERSTORM"}')
    print(f'Probability → No TH: {prob[0]:.3f}  |  TH: {prob[1]:.3f}')
else:
    print('No positive samples in test set. Using manual example.')
    sample_dict = {col: [X_test[col].mean()] for col in FEATURE_COLS}
    sample_df = pd.DataFrame(sample_dict)
    sample_scaled = loaded_scaler.transform(sample_df)
    pred = loaded_model.predict(sample_scaled)[0]
    print(f'Prediction: {"⛈️ THUNDERSTORM" if pred == 1 else "☀️ NO THUNDERSTORM"}')

## 🖥️ Step 11 — Write Streamlit App to Disk

In [ ]:
streamlit_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import os

# ── Paths ─────────────────────────────────────────────────────────
BASE_DIR    = '/content/thunderstorm_forecasting'
MODEL_PATH  = f'{BASE_DIR}/models/best_model_Random_Forest.pkl'
SCALER_PATH = f'{BASE_DIR}/models/scaler.pkl'

FEATURE_COLS = [
    'SWEAT index', 'Showalter index', 'LIFTED index', 'K index',
    'Cross totals index', 'Vertical totals index', 'Totals totals index',
    'TLCL', 'PLCL', 'CINE', 'CAPE', 'PRECIPITABLE WATER'
]

@st.cache_resource
def load_model_and_scaler():
    model  = joblib.load(MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)
    return model, scaler

model, scaler = load_model_and_scaler()

# ── UI ────────────────────────────────────────────────────────────
st.set_page_config(page_title='Thunderstorm Forecasting', page_icon='⛈️', layout='wide')
st.title('⛈️ Thunderstorm Forecasting System')
st.markdown('Enter atmospheric index values to predict thunderstorm occurrence.')

st.sidebar.header('📊 Input Atmospheric Indices')

defaults = {
    'SWEAT index': 80.0, 'Showalter index': 5.0, 'LIFTED index': 0.0,
    'K index': 20.0, 'Cross totals index': 20.0, 'Vertical totals index': 25.0,
    'Totals totals index': 45.0, 'TLCL': 280.0, 'PLCL': 900.0,
    'CINE': -10.0, 'CAPE': 100.0, 'PRECIPITABLE WATER': 30.0
}

input_vals = {}
for feat, default in defaults.items():
    input_vals[feat] = st.sidebar.number_input(
        feat, value=float(default), format='%.2f'
    )

if st.sidebar.button('🔍 Predict Thunderstorm', use_container_width=True):
    input_df = pd.DataFrame([input_vals])
    input_scaled = scaler.transform(input_df[FEATURE_COLS])
    pred = model.predict(input_scaled)[0]
    prob = model.predict_proba(input_scaled)[0]

    col1, col2 = st.columns(2)
    with col1:
        if pred == 1:
            st.error('⛈️ **THUNDERSTORM PREDICTED**')
        else:
            st.success('☀️ **NO THUNDERSTORM PREDICTED**')
    with col2:
        st.metric('Thunderstorm Probability', f"{prob[1]*100:.1f}%")
        st.metric('No-Thunderstorm Probability', f"{prob[0]*100:.1f}%")

    st.subheader('Input Summary')
    st.dataframe(input_df, use_container_width=True)

st.divider()
st.markdown('**Model:** Random Forest Classifier | **SMOTE** applied for class imbalance | Logged via **MLflow / DagsHub**')
'''

ui_path = f'{BASE_DIR}/streamlit_app/ui.py'
with open(ui_path, 'w') as f:
    f.write(streamlit_code)

print(f'✅ Streamlit app written → {ui_path}')

## 🔌 Step 12 — Write FastAPI Inference API

In [ ]:
api_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

BASE_DIR    = '/content/thunderstorm_forecasting'
model  = joblib.load(f'{BASE_DIR}/models/best_model_Random_Forest.pkl')
scaler = joblib.load(f'{BASE_DIR}/models/scaler.pkl')

FEATURE_COLS = [
    'sweat_index', 'showalter_index', 'lifted_index', 'k_index',
    'cross_totals', 'vertical_totals', 'totals_totals',
    'tlcl', 'plcl', 'cine', 'cape', 'precipitable_water'
]

app = FastAPI(title='Thunderstorm Forecasting API')

class FeaturesIn(BaseModel):
    sweat_index: float
    showalter_index: float
    lifted_index: float
    k_index: float
    cross_totals: float
    vertical_totals: float
    totals_totals: float
    tlcl: float
    plcl: float
    cine: float
    cape: float
    precipitable_water: float

@app.get("/")
def root():
    return {"message": "Thunderstorm Forecasting API — POST /predict"}

@app.post("/predict")
def predict(data: FeaturesIn):
    x = np.array([[getattr(data, col) for col in FEATURE_COLS]])
    x_scaled = scaler.transform(x)
    pred = int(model.predict(x_scaled)[0])
    prob = float(model.predict_proba(x_scaled)[0][1])
    label = "THUNDERSTORM" if pred == 1 else "NO THUNDERSTORM"
    return {"prediction": pred, "label": label, "probability_thunderstorm": prob}
'''

api_path = f'{BASE_DIR}/api/main.py'
with open(api_path, 'w') as f:
    f.write(api_code)

print(f'✅ FastAPI app written → {api_path}')

## 🚀 Step 13 — Launch Streamlit via ngrok
> Requires `NGROK_AUTHTOKEN` in Colab Secrets

In [ ]:
import subprocess, time
from pyngrok import ngrok
from google.colab import userdata

PORT = 8501

# ── Authenticate ngrok ───────────────────────────────────────────
ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))

# ── Start Streamlit in background ────────────────────────────────
process = subprocess.Popen(
    ['streamlit', 'run', f'{BASE_DIR}/streamlit_app/ui.py',
     '--server.port', str(PORT),
     '--server.headless', 'true',
     '--server.enableCORS', 'false'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

time.sleep(4)  # Wait for Streamlit to boot

# ── Open public tunnel ───────────────────────────────────────────
public_url = ngrok.connect(PORT)
print(f'\n🌐 Streamlit app LIVE at: {public_url}')
print('Share this URL to access the Thunderstorm Forecasting UI.')

## 📈 Step 14 — (Optional) Launch MLflow UI via ngrok
Only needed if `USE_DAGSHUB = False`. If you're using DagsHub, view runs at your DagsHub repo.

In [ ]:
# ── Only run this cell if USE_DAGSHUB = False ────────────────────
import subprocess, time
from pyngrok import ngrok

MLFLOW_PORT = 5000

mlflow_proc = subprocess.Popen(
    ['mlflow', 'ui', '--host', '0.0.0.0', '--port', str(MLFLOW_PORT)],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(3)
mlflow_url = ngrok.connect(MLFLOW_PORT)
print(f'\n📈 MLflow UI LIVE at: {mlflow_url}')

## 💾 Step 15 — Save All Outputs & Download as ZIP

In [ ]:
import shutil

# Save metrics CSV
results_df.to_csv(f'{BASE_DIR}/outputs/model_comparison.csv')

# Create zip of entire project
zip_path = '/content/thunderstorm_forecasting_colab'
shutil.make_archive(zip_path, 'zip', '/content', 'thunderstorm_forecasting')
print(f'✅ Project zipped → {zip_path}.zip')

# Download
from google.colab import files
files.download(f'{zip_path}.zip')
print('✅ Download triggered!')

## 📋 Project Summary

| Step | Description | Output |
|------|-------------|--------|
| 1 | Colab Secrets → Env vars | `MLFLOW_TRACKING_URI`, `NGROK_AUTHTOKEN` |
| 2 | Directory Setup | `/content/thunderstorm_forecasting/` |
| 3 | Load Raw CSVs | `indices data.csv`, `surface_data.csv` |
| 4 | EDA | Boxplots, heatmap, class distribution |
| 5 | Merge & Preprocess | `merged_df_all12k_combined.csv` |
| 6 | SMOTE + Scaling | Balanced training set |
| 7 | MLflow Tracking | RF, KNN, LR, DT — all logged |
| 8 | Best Model Save | `best_model_Random_Forest.pkl` |
| 9 | Streamlit App | Live URL via ngrok |
| 10 | ZIP Download | Full project saved locally |

**References:**
- App Demo: https://thunderstrom-forecast.streamlit.app/
- GitHub: https://github.com/d-hackmt/thunderstrom-forecast
- MLflow: https://mlflow.org/
- Streamlit Cloud: https://streamlit.io/cloud
- DagsHub: https://dagshub.com/